<a href="https://colab.research.google.com/github/Adrentur/-1_-_-_-3-10/blob/main/%D0%A2%D0%B5%D1%81%D1%82%D0%BE%D0%B2%D0%B5_%D0%B7%D0%B0%D0%B2%D0%B4%D0%B0%D0%BD%D0%BD%D1%8F_ML_%D0%B4%D0%B6%D1%83%D0%BD%D1%96%D0%BE%D1%80__%D0%9F%D0%BE%D0%BB%D1%94%D1%85%D1%96%D0%BD_%D0%84%D0%B3%D0%BE%D1%80_3_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd

urls = [
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv",
    "https://raw.githubusercontent.com/plotly/datasets/master/telco-customer-churn.csv",
]

df = None
for u in urls:
    try:
        df = pd.read_csv(u)
        break
    except Exception:
        pass

if df is None:
    rng = np.random.default_rng(42)
    n = 2000
    tenure = rng.integers(1, 72, size=n)
    monthly = np.round(rng.uniform(20, 115, size=n), 2)
    contract = rng.choice(["Month-to-month", "One year", "Two year"], size=n, p=[0.55, 0.25, 0.20])
    churn = np.where(
        rng.random(n) < (1 / (1 + np.exp(-(-1.0 - 0.03 * tenure + (contract == "Month-to-month") * 1.0)))),
        "Yes",
        "No",
    )
    df = pd.DataFrame({
        "tenure": tenure,
        "Contract": contract,
        "MonthlyCharges": monthly,
        "TotalCharges": (tenure * monthly).round(2),
        "Churn": churn,
    })

df = df.drop(columns=["customerID"], errors="ignore")
print(df.shape)
print(df["Churn"].value_counts(normalize=True).round(2))

(7043, 20)
Churn
No     0.73
Yes    0.27
Name: proportion, dtype: float64


In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop(columns=["Churn"])
y = df["Churn"]

prep = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), X.select_dtypes(include="object").columns),
    ("num", "passthrough", X.select_dtypes(exclude="object").columns)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
candidates = {
    "LogReg": LogisticRegression(max_iter=2000, C=0.5),
    "RF": RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42),
    "GB": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
}

best_name, best_model, best_auc = None, None, -1

for name, clf in candidates.items():
    pipe = Pipeline([("prep", prep), ("clf", clf)]).fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = pipe.predict(X_test)
    auc = roc_auc_score(y_test, proba)
    print(f"{name}: acc={accuracy_score(y_test, pred):.3f}, auc={auc:.3f}")
    if auc > best_auc:
        best_name, best_model, best_auc = name, pipe, auc

print("Найкраща модель:", best_name)

LogReg: acc=0.805, auc=0.836
RF: acc=0.790, auc=0.835
GB: acc=0.795, auc=0.838
Найкраща модель: GB


In [6]:
client = X_test.iloc[[0]].copy()
client["tenure"] = 2
client["Contract"] = "Month-to-month"
client["MonthlyCharges"] = 85.0

res = best_model.predict(client)[0]
prob = best_model.predict_proba(client)[0][1]
print("Піде" if res == 1 else "Залишиться", f"({prob:.1%})")

Піде (58.8%)
